In [ ]:
# Colab-ready installs. Skip if you already have these packages.
# This may take a few minutes.
!pip install -q "transformers==4.56.2" tokenizers trl==0.22.2 datasets unsloth_zoo unsloth bitsandbytes accelerate scikit-learn pyarrow==19.0.0
# If you see GPU/CUDA related errors, try restarting the runtime after install.

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.1/40.1 kB 3.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.5/61.5 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 86.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 544.8/544.8 kB 51.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.1/42.1 MB 11.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 273.6/273.6 kB 29.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 45.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 348.7/348.7 kB 36.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.4/59.4 MB 18.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.2/7.2 MB 116.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.2/117.2 MB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 213.6/213.6 kB 24.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
import os
import math
from pathlib import Path
from tqdm import tqdm

import torch
from datasets import load_dataset

from sklearn.metrics import accuracy_score, f1_score, classification_report

print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)


torch: 2.8.0+cu126
cuda available: True
Using device: cuda


In [ ]:
# Login to HF if private dataset: run `huggingface-cli login` in a cell or terminal beforehand.
ds = load_dataset("DaniilOr/SemEval-2026-Task13", "A")   # per your note

# ds could be a DatasetDict or a Dataset. Normalize to train/validation splits.
if isinstance(ds, dict) or hasattr(ds, "keys"):
    keys = list(ds.keys())
    print("Dataset keys:", keys)
    if "train" in ds:
        train_ds = ds["train"]
    else:
        # if single split present, use it and create a train/validation split
        if len(keys) == 1:
            train_ds = ds[keys[0]].train_test_split(test_size=0.1, seed=3407)["train"]
            val_ds = ds[keys[0]].train_test_split(test_size=0.1, seed=3407)["test"]
        else:
            train_ds = ds[keys[0]]

    if "validation" in ds:
        val_ds = ds["validation"]
    else:
        # fallback: if user provided single dataset and we didn't already split
        if "val_ds" not in globals():
            val_ds = train_ds.train_test_split(test_size=0.1, seed=3407)["test"]

    if "test" in ds:
        test_ds = ds["test"]
else:
    # single dataset object
    full = ds
    split = full.train_test_split(test_size=0.1, seed=3407)
    train_ds = split["train"]
    val_ds = split["test"]

print("train size:", len(train_ds), "val size:", len(val_ds))
print("Columns:", train_ds.column_names)
# Quick peek
print(train_ds[0])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/801 [00:00<?, ?B/s]

task_a/task_a_training_set_1.parquet:   0%|          | 0.00/203M [00:00<?, ?B/s]

task_a/task_a_validation_set.parquet:   0%|          | 0.00/40.5M [00:00<?, ?B/s]

task_a/task_a_test_set_sample.parquet:   0%|          | 0.00/593k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/500000 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/100000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1000 [00:00<?, ? examples/s]

Dataset keys: ['train', 'validation', 'test']
train size: 500000 val size: 100000
Columns: ['code', 'generator', 'label', 'language']
{'code': "(a, b, c, d) = [int(x) for x in input().split()]\nk = input()\n(p, q, r, s) = (0, 0, 0, 0)\nfor i in k:\n\tif i == '1':\n\t\tp += 1\n\telif i == '2':\n\t\tq += 1\n\telif i == '3':\n\t\tr += 1\n\telif i == '4':\n\t\ts += 1\nprint(a * p + b * q + c * r + d * s)\n", 'generator': 'human', 'label': 0, 'language': 'Python'}


In [ ]:
from unsloth import FastLanguageModel

# Choose model — 4-bit prequantized recommended on Colab for memory reasons:
MODEL_NAME = "unsloth/gpt-oss-20b"  # change if you prefer another
MAX_SEQ_LENGTH = 1024
LOAD_IN_4BIT = True  # set False if you want full precision (requires much more VRAM)

print("Loading model/tokenizer (this downloads weights)...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = MODEL_NAME,
    dtype = None,
    max_seq_length = MAX_SEQ_LENGTH,
    load_in_4bit = LOAD_IN_4BIT,
    full_finetuning = False,  # we will use PEFT/LoRA below
    offload_embedding = True,
)
print("Model + tokenizer loaded.")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


🦥 Unsloth Zoo will now patch everything to make training faster!
Loading model/tokenizer (this downloads weights)...
==((====))==  Unsloth 2025.10.12: Fast Gpt_Oss patching. Transformers: 4.56.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.8.0+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.4.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.32.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Using float16 precision for gpt_oss won't work! Using float32.


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

model-00001-of-00004.safetensors:   0%|          | 0.00/4.00G [00:00<?, ?B/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/27.9M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/446 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

Model + tokenizer loaded.


In [ ]:
def build_texts(batch):
    codes = batch.get("code", [])
    labels = batch.get("label", [])
    langs = batch.get("language", [])
    out_texts = []
    for i, code in enumerate(codes):
        label = labels[i] if i < len(labels) else 0
        lang = langs[i] if i < len(langs) else "Unknown"
        prompt = f"""
You are a code origin classifier. Your task is to determine whether the following code was written by a human or generated by a machine.

Output rule:
- Respond with EXACTLY ONE WORD: "human" or "machine".
- Do not include any explanations, punctuation, or extra text.

Guidelines for reasoning:
1. Naming: Human code shows mixed styles or domain-specific names. Machine code uses consistent, verbose, or generic names.
2. Comments: Human comments are sparse or informal. Machine comments are detailed, redundant, or uniformly formatted.
3. Structure: Human code may have shortcuts or irregularities. Machine code is overly structured or templated.
4. Formatting: Human code has minor inconsistencies. Machine code follows perfect style rules.
5. Logic: Human logic is pragmatic and iterative. Machine logic is overly complete or formal.

Language: {lang}

Code:
```{lang.lower()}
{code}
Output your decision (human or machine):
"""

        assistant_reply = "machine" if int(label) == 1 else "human"
        convo = [
            {"role": "user", "content": prompt},
            {"role": "assistant", "content": assistant_reply},
        ]
        text = tokenizer.apply_chat_template(convo, tokenize=False, add_generation_prompt=False)
        out_texts.append(text)
    return {"text": out_texts}

# Apply mapping (batched). This will add a `text` column that SFTTrainer expects.
train_ds = train_ds.map(build_texts, batched=True, batch_size=128)
val_ds = val_ds.map(build_texts, batched=True, batch_size=128)

print("Formatting done. Example formatted text (train):")
print(train_ds[0]["text"][:1000].replace("\n", "\\n")[:1000])

Map:   0%|          | 0/500000 [00:00<?, ? examples/s]

Map:   0%|          | 0/100000 [00:00<?, ? examples/s]

Formatting done. Example formatted text (train):
<|start|>system<|message|>You are ChatGPT, a large language model trained by OpenAI.\nKnowledge cutoff: 2024-06\nCurrent date: 2025-11-03\n\nReasoning: medium\n\n# Valid channels: analysis, commentary, final. Channel must be included for every message.\nCalls to these tools must go to the commentary channel: 'functions'.<|end|><|start|>user<|message|>\nYou are a code origin classifier. Your task is to determine whether the following code was written by a human or generated by a machine.\n\nOutput rule:\n- Respond with EXACTLY ONE WORD: "human" or "machine".\n- Do not include any explanations, punctuation, or extra text.\n\nGuidelines for reasoning:\n1. Naming: Human code shows mixed styles or domain-specific names. Machine code uses consistent, verbose, or generic names.\n2. Comments: Human comments are sparse or informal. Machine comments are detailed, redundant, or uniformly formatted.\n3. Structure: Human code may have shortcuts or ir

In [ ]:
print("\n" + "="*80)
print("CHECKING CHAT TEMPLATE FORMAT")
print("="*80)
example_text = train_ds[0]["text"]
print("First 1500 characters of formatted example:")
print(example_text[:6000])
print("\n" + "="*80)
print("IMPORTANT: Look at the output above and identify the EXACT markers")
print("for where the assistant response starts. Update gpt_oss_kwargs below.")
print("="*80 + "\n")


CHECKING CHAT TEMPLATE FORMAT
First 1500 characters of formatted example:
<|start|>system<|message|>You are ChatGPT, a large language model trained by OpenAI.
Knowledge cutoff: 2024-06
Current date: 2025-11-03

Reasoning: medium

# Valid channels: analysis, commentary, final. Channel must be included for every message.
Calls to these tools must go to the commentary channel: 'functions'.<|end|><|start|>user<|message|>
You are a code origin classifier. Your task is to determine whether the following code was written by a human or generated by a machine.

Output rule:
- Respond with EXACTLY ONE WORD: "human" or "machine".
- Do not include any explanations, punctuation, or extra text.

Guidelines for reasoning:
1. Naming: Human code shows mixed styles or domain-specific names. Machine code uses consistent, verbose, or generic names.
2. Comments: Human comments are sparse or informal. Machine comments are detailed, redundant, or uniformly formatted.
3. Structure: Human code may have shortc

In [ ]:
# Attach PEFT/LoRA adapters to the loaded model (only a small fraction of params will be trained)
model = FastLanguageModel.get_peft_model(
    model,
    r = 8,  # try 8-32 depending on capacity
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0.0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)

print("PEFT/LoRA adapters attached.")


Unsloth: Making `model.base_model.model.model` require gradients
PEFT/LoRA adapters attached.


In [ ]:
from trl import SFTConfig, SFTTrainer
from unsloth.chat_templates import train_on_responses_only

sft_args = SFTConfig(
    per_device_train_batch_size = 1,   # lower for Colab; increase for bigger GPUs
    gradient_accumulation_steps = 4,
    warmup_steps = 5,
    max_steps = 500,
    # num_train_epochs = 1,              # small demo run; set higher or use num_train_epochs
    learning_rate = 2e-4,
    logging_steps = 10,
    optim = "adamw_8bit",
    weight_decay = 0.01,
    lr_scheduler_type = "linear",
    seed = 3407,
    output_dir = "outputs",
    report_to = "none",
)

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_ds,
    args = sft_args,
)

# Use the correct assistant marker that appears in your formatted text
gpt_oss_kwargs = dict(
    instruction_part = "<|start|>user<|message|>",
    response_part    = "<|start|>assistant<|message|>",   # <-- corrected
)

trainer = train_on_responses_only(trainer, **gpt_oss_kwargs)


print("\n" + "="*80)
print("MASKING VERIFICATION")
print("="*80)

def verify_masking(trainer, indices=[0, 10, min(100, len(trainer.train_dataset)-1)]):
    """Verify that only assistant responses are being trained on."""
    for idx in indices:
        if idx >= len(trainer.train_dataset):
            continue

        print(f"\n--- Example {idx} ---")

        # Show full input
        input_ids = trainer.train_dataset[idx]["input_ids"]
        full_text = tokenizer.decode(input_ids)
        print("Full conversation (last 300 chars):")
        print(full_text[-300:])

        # Show what's being trained on (should be ONLY "human" or "machine")
        labels = trainer.train_dataset[idx]["labels"]
        trained_tokens = [
            tokenizer.pad_token_id if x == -100 else x
            for x in labels
        ]
        trained_text = tokenizer.decode(trained_tokens).replace(
            tokenizer.pad_token, " "
        ).strip()

        print("\nWhat model is trained on (should be ONLY 'human' or 'machine'):")
        print(f"'{trained_text}'")

        non_masked_count = sum(1 for x in labels if x != -100)
        total_count = len(labels)
        print(f"Non-masked tokens: {non_masked_count}/{total_count}")

        # CRITICAL CHECK: Should be 1-5 tokens max (just "human" or "machine")
        if non_masked_count > 10:
            print("⚠️  WARNING: Too many non-masked tokens! Masking may not be working.")
            print("   Expected: 1-5 tokens (just the answer)")
            print(f"   Got: {non_masked_count} tokens")
        else:
            print("✓ Masking looks correct!")

verify_masking(trainer)

Unsloth: Switching to float32 training since model cannot work with float16


Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/500000 [00:00<?, ? examples/s]

Map (num_proc=6):   0%|          | 0/500000 [00:00<?, ? examples/s]


MASKING VERIFICATION

--- Example 0 ---
Full conversation (last 300 chars):
in input().split()]
k = input()
(p, q, r, s) = (0, 0, 0, 0)
for i in k:
	if i == '1':
		p += 1
	elif i == '2':
		q += 1
	elif i == '3':
		r += 1
	elif i == '4':
		s += 1
print(a * p + b * q + c * r + d * s)

Output your decision (human or machine):
<|end|><|start|>assistant<|message|>human<|return|>

What model is trained on (should be ONLY 'human' or 'machine'):
'human<|return|>'
Non-masked tokens: 2/386
✓ Masking looks correct!

--- Example 10 ---
Full conversation (last 300 chars):
thon

Code:
```python
a = sorted(list(map(int, input().split())), reverse=True)
f1 = a[0] + a[3]
f2 = a[1] + a[2]
if f1 == f2:
	print('YES')
elif a[0] == a[1] + a[2] + a[3]:
	print('YES')
else:
	print('NO')

Output your decision (human or machine):
<|end|><|start|>assistant<|message|>human<|return|>

What model is trained on (should be ONLY 'human' or 'machine'):
'human<|return|>'
Non-masked tokens: 2/351
✓ Masking looks correct!

In [ ]:
def count_examples_with_labels(ds, sample_n=2000):
    # checks first sample_n examples (or entire ds if smaller)
    N = min(len(ds), sample_n)
    has = 0
    for i in range(N):
        lbls = ds[i]['labels']
        if any([x != -100 for x in lbls]):
            has += 1
    return has, N

has, total = count_examples_with_labels(trainer.train_dataset, sample_n=2000)
print(f"{has}/{total} examples have at least one non- -100 label.")

In [ ]:
print("Starting training...")
trainer_stats = trainer.train()
print("Training finished.")
print(trainer_stats)


The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': 199998, 'pad_token_id': 200017}.


Starting training...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 2
   \\   /|    Num examples = 500,000 | Num Epochs = 1 | Total steps = 500
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 4 x 1) = 4
 "-____-"     Trainable parameters = 3,981,312 of 20,918,738,496 (0.02% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
10,5.339900
20,0.440100
30,0.449500
40,0.195500
50,0.109000
60,0.000200
70,0.520500
80,0.269400
90,0.008800
100,0.387800


Training finished.
TrainOutput(global_step=500, training_loss=0.21067804431752302, metrics={'train_runtime': 9541.0145, 'train_samples_per_second': 0.21, 'train_steps_per_second': 0.052, 'total_flos': 1.236180176488512e+17, 'train_loss': 0.21067804431752302, 'epoch': 0.004})


In [ ]:
SAVE_DIR = "finetuned_gptoss_lora_2"
print("Saving adapters to", SAVE_DIR)
model.save_pretrained(SAVE_DIR)
print("Saved.")
# Optionally push to hub: model.push_to_hub("hf_username/repo", token="hf_...")


Saving adapters to finetuned_gptoss_lora_2
Saved.


In [ ]:
!zip -r /content/finetuned_gptoss_lora_2.zip /content/finetuned_gptoss_lora_2

  adding: content/finetuned_gptoss_lora_2/ (stored 0%)
  adding: content/finetuned_gptoss_lora_2/adapter_config.json (deflated 57%)
  adding: content/finetuned_gptoss_lora_2/adapter_model.safetensors (deflated 7%)
  adding: content/finetuned_gptoss_lora_2/README.md (deflated 65%)


In [ ]:
from google.colab import files

files.download('finetuned_gptoss_lora_2.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

### Evaluation:

In [ ]:
!unzip -q /content/outputs.zip -d /content/outputs

[/content/outputs.zip]
  End-of-central-directory signature not found.  Either this file is not
  a zipfile, or it constitutes one disk of a multi-part archive.  In the
  latter case the central directory and zipfile comment will be found on
  the last disk(s) of this archive.
unzip:  cannot find zipfile directory in one of /content/outputs.zip or
        /content/outputs.zip.zip, and cannot find /content/outputs.zip.ZIP, period.


In [ ]:
!unzip -q /content/finetuned_gptoss_lora_2.zip -d /

In [ ]:
!zip -r /content/outputs.zip /content/outputs

  adding: content/outputs/ (stored 0%)
  adding: content/outputs/README.md (deflated 44%)
  adding: content/outputs/checkpoint-200/ (stored 0%)
  adding: content/outputs/checkpoint-200/training_args.bin (deflated 53%)
  adding: content/outputs/checkpoint-200/trainer_state.json (deflated 74%)
  adding: content/outputs/checkpoint-200/adapter_model.safetensors (deflated 7%)
  adding: content/outputs/checkpoint-200/README.md (deflated 65%)
  adding: content/outputs/checkpoint-200/tokenizer_config.json (deflated 89%)
  adding: content/outputs/checkpoint-200/rng_state.pth (deflated 26%)
  adding: content/outputs/checkpoint-200/scheduler.pt (deflated 62%)
  adding: content/outputs/checkpoint-200/chat_template.jinja (deflated 75%)
  adding: content/outputs/checkpoint-200/tokenizer.json (deflated 84%)
  adding: content/outputs/checkpoint-200/adapter_config.json (deflated 57%)
  adding: content/outputs/checkpoint-200/optimizer.pt (deflated 9%)
  adding: content/outputs/checkpoint-200/special_tok

In [ ]:
!pip install -q "transformers==4.56.2" tokenizers datasets unsloth_zoo unsloth bitsandbytes accelerate scikit-learn pyarrow==19.0.0

import torch
from datasets import load_dataset
from unsloth import FastLanguageModel
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
import numpy as np
from tqdm import tqdm

print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.1/40.1 kB 1.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.5/61.5 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 39.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.1/42.1 MB 19.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 276.7/276.7 kB 12.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 20.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 348.8/348.8 kB 17.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.4/59.4 MB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.2/7.2 MB 34.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 564.7/564.7 kB 14.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.2/117.2 MB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 213.6/213.6 kB 14.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
BASE_MODEL = "unsloth/gpt-oss-20b"  # same as training
ADAPTER_PATH = "finetuned_gptoss_lora_2"  # path to your saved adapters
MAX_SEQ_LENGTH = 1024
LOAD_IN_4BIT = True

print("Loading base model + adapters...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = ADAPTER_PATH,  # this loads base model + adapters
    dtype = None,
    max_seq_length = MAX_SEQ_LENGTH,
    load_in_4bit = LOAD_IN_4BIT,
)

# Enable inference mode for faster generation
FastLanguageModel.for_inference(model)
print("Model loaded and ready for inference.")

Loading base model + adapters...
==((====))==  Unsloth 2025.11.1: Fast Gpt_Oss patching. Transformers: 4.56.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.8.0+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.4.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.32.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Using float16 precision for gpt_oss won't work! Using float32.


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

model-00003-of-00004.safetensors:   0%|          | 0.00/3.37G [00:00<?, ?B/s]

model-00002-of-00004.safetensors:   0%|          | 0.00/4.00G [00:00<?, ?B/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/27.9M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/446 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

Model loaded and ready for inference.


In [ ]:
ds = load_dataset("DaniilOr/SemEval-2026-Task13", "A")

# Get validation split (same logic as training)
if isinstance(ds, dict) or hasattr(ds, "keys"):
    keys = list(ds.keys())
    print("Dataset keys:", keys)
    # if "validation" in ds:
    #     val_ds = ds["validation"]
    if "test" in ds:
        test_ds = ds["test"]
else:
    val_ds = ds.train_test_split(test_size=0.1, seed=3407)["test"]

print(f"Test set size: {len(test_ds)}")

README.md:   0%|          | 0.00/801 [00:00<?, ?B/s]

task_a/task_a_training_set_1.parquet:   0%|          | 0.00/203M [00:00<?, ?B/s]

task_a/task_a_validation_set.parquet:   0%|          | 0.00/40.5M [00:00<?, ?B/s]

task_a/task_a_test_set_sample.parquet:   0%|          | 0.00/593k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/500000 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/100000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1000 [00:00<?, ? examples/s]

Dataset keys: ['train', 'validation', 'test']
Test set size: 1000


In [ ]:
test_ds

Dataset({
    features: ['code', 'generator', 'label', 'language'],
    num_rows: 1000
})

In [ ]:
MAX_CODE_CHARS = 5000

def _crop_head_tail(text: str, max_chars: int) -> str:
    if len(text) <= max_chars:
        return text
    half = max_chars // 2
    return text[:half] + "\n... [truncated] ...\n" + text[-half:]

def _build_prompt(lang: str, code: str) -> str:
    # Use the SAME prompt structure as training
    prompt = f"""
            You are a code origin classifier. Your task is to determine whether the following code was written by a human or generated by a machine.

            Output rule:
            - Respond with EXACTLY ONE WORD: "human" or "machine".
            - Do not include any explanations, punctuation, or extra text.

            Guidelines for reasoning:
            1. Naming: Human code shows mixed styles or domain-specific names. Machine code uses consistent, verbose, or generic names.
            2. Comments: Human comments are sparse or informal. Machine comments are detailed, redundant, or uniformly formatted.
            3. Structure: Human code may have shortcuts or irregularities. Machine code is overly structured or templated.
            4. Formatting: Human code has minor inconsistencies. Machine code follows perfect style rules.
            5. Logic: Human logic is pragmatic and iterative. Machine logic is overly complete or formal.

            Language: {lang}

            Code:
            ```{lang.lower()}
            {code}
            Output your decision (human or machine):
            """
    return prompt

def predict_single(code, language="Unknown"):
    """
    Predict whether code is human or machine-generated.
    Returns: "human" or "machine"
    """
    # Crop code if too long
    code_snippet = _crop_head_tail(code, MAX_CODE_CHARS)

    # Build prompt
    prompt = _build_prompt(language, code_snippet)

    # Format as messages
    messages = [{"role": "user", "content": prompt}]

    # Apply chat template WITHOUT tokenizing (returns string)
    input_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,  # Returns string, not tensors
        add_generation_prompt=True
    )

    # Tokenize with explicit truncation (CRITICAL for memory)
    inputs = tokenizer(
        input_text,
        return_tensors="pt",
        truncation=True,  # Truncate if too long
        max_length=MAX_SEQ_LENGTH  # Hard limit
    )

    # Move to GPU
    inputs = {k: v.to("cuda") for k, v in inputs.items()}

    # Generate
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=5,  # Only need 1 word
            temperature=0.1,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    # Decode only the generated tokens
    input_length = inputs['input_ids'].shape[1]
    generated_tokens = outputs[0][input_length:]
    generated_text = tokenizer.decode(generated_tokens, skip_special_tokens=True)
    response_text = generated_text.strip().lower()

    # Parse response
    if "machine" in response_text:
        return "machine"
    elif "human" in response_text:
        return "human"
    else:
        first_word = response_text.split()[0] if response_text.split() else ""
        if first_word in ["human", "machine"]:
            return first_word
        print(f"Warning: Unexpected output: '{response_text[:100]}'")
        return "human"

In [ ]:
torch.cuda.empty_cache()
import gc
gc.collect()

predictions = []
true_labels = []
raw_outputs = []

# Now run on full test set (no verbose to avoid clutter)
for idx in tqdm(range(len(test_ds)), desc="Predicting"):
    sample = test_ds[idx]
    code = sample.get("code", "")
    label = sample.get("label", 0)
    lang = sample.get("language", "Unknown")

    # Get prediction
    pred_text = predict_single(code, lang)
    raw_outputs.append(pred_text)

    # Convert to binary label
    pred_label = 1 if pred_text == "machine" else 0

    predictions.append(pred_label)
    true_labels.append(int(label))

    # Periodic memory cleanup
    if idx % 50 == 0 and idx > 0:
        torch.cuda.empty_cache()
        gc.collect()

# Validation
unexpected = sum(x not in ("human", "machine") for x in raw_outputs)
print(f"\nNon-binary outputs: {unexpected} (should be 0)")

if unexpected > 0:
    print("\nUnexpected outputs:")
    for i, out in enumerate(raw_outputs):
        if out not in ("human", "machine"):
            print(f"  Index {i}: '{out}'")

Predicting:   0%|          | 3/1000 [01:37<6:37:33, 23.93s/it] 

Predicting:   0%|          | 4/1000 [01:44<4:47:29, 17.32s/it]

Predicting:   0%|          | 5/1000 [01:51<3:46:32, 13.66s/it]

      return true'


Predicting:   1%|          | 8/1000 [02:07<2:14:44,  8.15s/it]

...

Predicting:  99%|█████████▊| 986/1000 [1:11:38<01:05,  4.70s/it]

        $'


Predicting:  99%|█████████▉| 989/1000 [1:11:57<01:05,  5.99s/it]

Predicting: 100%|██████████| 1000/1000 [1:12:42<00:00,  4.36s/it]


Non-binary outputs: 0 (should be 0)


In [ ]:
# Show some examples
print("\nFirst 10 predictions:")
for i in range(min(10, len(raw_outputs))):
    correct = "✓" if predictions[i] == true_labels[i] else "✗"
    print(f"  {correct} {i}: Predicted={raw_outputs[i]}, True={'machine' if true_labels[i] == 1 else 'human'}")

# Calculate accuracy
from sklearn.metrics import accuracy_score, classification_report
accuracy = accuracy_score(true_labels, predictions)
print(f"\nAccuracy: {accuracy:.4f}")
print("\nClassification Report:")
print(classification_report(true_labels, predictions, target_names=["human", "machine"]))


First 10 predictions:
  ✗ 0: Predicted=machine, True=human
  ✓ 1: Predicted=human, True=human
  ✓ 2: Predicted=human, True=human
  ✓ 3: Predicted=human, True=human
  ✓ 4: Predicted=human, True=human
  ✗ 5: Predicted=machine, True=human
  ✓ 6: Predicted=human, True=human
  ✓ 7: Predicted=human, True=human
  ✓ 8: Predicted=machine, True=machine
  ✗ 9: Predicted=machine, True=human

Accuracy: 0.5860

Classification Report:
              precision    recall  f1-score   support

       human       0.90      0.52      0.66       777
     machine       0.33      0.80      0.46       223

    accuracy                           0.59      1000
   macro avg       0.61      0.66      0.56      1000
weighted avg       0.77      0.59      0.62      1000



In [ ]:
import pandas as pd

df = pd.DataFrame({
    'ID': range(len(predictions)),
    'label': predictions
})

df.to_csv('test_predictions.csv', index=False)
print(f"CSV file created with {len(df)} rows")

CSV file created with 1000 rows


In [ ]:
from google.colab import files

files.download('test_predictions.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>